In [12]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\sendim\eletromec"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\sendim\eletromec\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Sendim - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Sendim - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Sendim - Eletromecânico.dwg")

conn = load_to_duckdb(dxf_file, "PV Sendim", rewrite=True)
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Sendim already exists
Removed existing data for project: PV Sendim
Successfully loaded project
Loaded 6033 entities for project: PV Sendim
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [13]:
data = conn.sql("SELECT layer, name, X, Y, UFV,PST, INV, STR, GRP FROM cad_entities WHERE project_name = 'PV Sendim'")
print(data)

┌────────────────────────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│         layer          │        name         │     X     │     Y      │   UFV   │   PST   │   INV   │   STR   │   GRP   │
│        varchar         │       varchar       │   float   │   float    │ varchar │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────────────┼───────────┼────────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ 0                      │ PV Sendim - Civil   │       0.0 │        0.0 │ NULL    │ NULL    │ NULL    │ NULL    │ NULL    │
│ EMF_em_ufv_Sendim_01.1 │ EMF302_eletrocentro │  49066.43 │   155151.3 │ 1       │ 1       │ NULL    │ NULL    │ NULL    │
│ EMF_em_ufv_Sendim_01.2 │ EMF302_eletrocentro │  49526.32 │ 155042.125 │ 1       │ 2       │ NULL    │ NULL    │ NULL    │
│ EMF_em_ufv_Sendim_01.3 │ EMF302_eletrocentro │  49666.48 │  154873.33 │ 1       │ 3       │ NULL    │ NULL    │ NULL    │
│ EMF_em

In [14]:
strings = conn.sql("SELECT layer, X, Y, UFV, PST, INV, STR FROM data WHERE name LIKE 'EMF302_string_%'")
strings_df = strings.fetchdf()
strings_df['TAG'] = "SND.STR-" + strings_df['UFV'].astype(str) + "." + strings_df['PST'].astype(str) + "." + strings_df['INV'].astype(str).str.zfill(2) + "." + strings_df['STR'].astype(str).str.zfill(2).str.zfill(2)
print(strings_df[['X', 'Y', 'TAG']])

                 X              Y                TAG
0     48950.972656  155332.015625  SND.STR-1.1.01.01
1     48960.894531  155325.687500  SND.STR-1.1.01.02
2     48960.894531  155323.375000  SND.STR-1.1.01.03
3     48960.894531  155318.187500  SND.STR-1.1.01.04
4     48960.894531  155315.875000  SND.STR-1.1.01.05
...            ...            ...                ...
5632  49760.644531  156177.015625  SND.STR-8.1.01.08
5633  49760.644531  156174.687500  SND.STR-8.1.01.09
5634  49791.070312  156184.515625  SND.STR-8.1.01.02
5635  49791.070312  156182.187500  SND.STR-8.1.01.03
5636  49852.445312  156190.843750  SND.STR-8.1.01.01

[5637 rows x 3 columns]


In [15]:
strings_df['Script'] = "(command-s \"._mtext\" \"" + strings_df['X'].astype(str) + "," + \
                     strings_df['Y'].astype(str) + \
                     "\" \"h\" 1.5  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     strings_df['TAG'].astype(str) + \
                     "\" \"\")"

strings_df['layer TAG'] = "EMF_tag_ufv_" + strings_df['UFV'].astype(str) + "." + strings_df['PST'].astype(str)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Inversor e Estruturas\Sendim - Script Estrutura.scr", 'w') as f:
    for i in strings_df['layer TAG'].unique():
        mask = strings_df['layer TAG'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(strings_df[mask]['Script'].astype(str)) + '\n')

In [16]:
conn.close()